# Phase 2 — Embeddings, FAISS, BM25, and Hybrid Retrieval




## Cell 1 — Locate your project


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "data" / "processed" / "knowledge_base_chunks.jsonl").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/processed/knowledge_base_chunks.jsonl. "
        "Save this notebook in the project's notebooks folder and confirm Phase 1 completed."
    )

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

Project root: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents
Python executable: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Scripts\python.exe


## Cell 2 — Install libraries



In [2]:
%pip install -U sentence-transformers faiss-cpu rank-bm25 pandas

  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)


## Cell 3 — Import libraries

In [3]:
import json
import re
import pickle
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

print("Imports successful.")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.
NumPy: 2.5.4
Pandas: 3.0.6


## Cell 4 — Load Phase 1 chunks

In [4]:
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "knowledge_base_chunks.jsonl"

chunks = []
with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if line.strip():
            try:
                chunks.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_number}: {exc}") from exc

chunks_df = pd.DataFrame(chunks)
print("Number of chunks:", len(chunks_df))
print("Columns:", list(chunks_df.columns))
display(chunks_df.head())
if chunks_df.empty:
    raise ValueError("The chunk file is empty. Re-run Phase 1 first.")

Number of chunks: 18
Columns: ['chunk_id', 'document_id', 'title', 'category', 'source', 'chunk_index', 'text', 'word_count']


,chunk_id,document_id,title,category,source,chunk_index,text,word_count
0,FAQ001_CHUNK001,FAQ001,Track an order,orders,sample_faq,1,Customers can check an order's status from the...,45
1,FAQ002_CHUNK001,FAQ002,Cancel an order,orders,sample_faq,1,An order can usually be cancelled while it is ...,49
2,POL001_CHUNK001,POL001,Shipping and delivery policy,shipping,sample_policy,1,Standard delivery typically takes 3 to 7 busin...,51
3,FAQ003_CHUNK001,FAQ003,Change a delivery address,shipping,sample_faq,1,Customers should verify their delivery address...,51
4,POL002_CHUNK001,POL002,Returns policy,returns,sample_policy,1,Eligible items may be returned within 30 calen...,62


## Cell 5 — Identify the text and ID fields



In [5]:
TEXT_FIELD_CANDIDATES = ["chunk_text", "text", "content", "page_content", "cleaned_text"]
ID_FIELD_CANDIDATES = ["chunk_id", "id", "chunkId"]

TEXT_FIELD = next((c for c in TEXT_FIELD_CANDIDATES if c in chunks_df.columns), None)
if TEXT_FIELD is None:
    raise KeyError(f"No recognized text column. Available columns: {list(chunks_df.columns)}")

ID_FIELD = next((c for c in ID_FIELD_CANDIDATES if c in chunks_df.columns), None)
if ID_FIELD is None:
    chunks_df["generated_chunk_id"] = [f"chunk_{i:04d}" for i in range(len(chunks_df))]
    ID_FIELD = "generated_chunk_id"

chunks_df[TEXT_FIELD] = chunks_df[TEXT_FIELD].fillna("").astype(str).str.strip()
chunks_df = chunks_df[chunks_df[TEXT_FIELD].ne("")].reset_index(drop=True)

texts = chunks_df[TEXT_FIELD].tolist()
chunk_ids = chunks_df[ID_FIELD].astype(str).tolist()

if not texts:
    raise ValueError("No non-empty chunk text was found.")

print("Text field:", TEXT_FIELD)
print("ID field:", ID_FIELD)
print("Non-empty chunks:", len(texts))
print("\nExample chunk:\n", texts[0][:700])

Text field: text
ID field: chunk_id
Non-empty chunks: 18

Example chunk:
 Customers can check an order's status from the Orders page after signing in. Open the order to see its current status, estimated delivery date, and tracking link when available. If tracking has not updated for more than 48 hours, contact support with the order number.


## Cell 6 — Load a compact embedding model



In [6]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Loaded:", EMBEDDING_MODEL_NAME)

c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tanish.Ranjan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Deskto

Loaded: sentence-transformers/all-MiniLM-L6-v2


## Cell 7 — Create normalized embeddings

In [7]:
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("Embedding matrix shape:", embeddings.shape)
print("Chunk count:", len(texts))
print("Embedding dimension:", embeddings.shape[1])
print("All values finite:", bool(np.isfinite(embeddings).all()))

assert embeddings.shape[0] == len(texts), "Embedding count does not match chunk count."
assert np.isfinite(embeddings).all(), "Embeddings contain invalid numeric values."

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

Embedding matrix shape: (18, 384)
Chunk count: 18
Embedding dimension: 384
All values finite: True


## Cell 8 — Build the FAISS semantic-search index



In [8]:
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print("FAISS index created.")
print("Vectors in index:", faiss_index.ntotal)
assert faiss_index.ntotal == len(texts), "FAISS index count mismatch."

FAISS index created.
Vectors in index: 18


## Cell 9 — Build the BM25 keyword-search index

In [9]:
def tokenize(text):
    """Convert text into lowercase word tokens."""
    return re.findall(r"[a-z0-9]+", str(text).lower())

tokenized_chunks = [tokenize(text) for text in texts]
bm25_index = BM25Okapi(tokenized_chunks)

print("BM25 index built for", len(tokenized_chunks), "chunks.")
print("Example tokens:", tokenized_chunks[0][:25])

BM25 index built for 18 chunks.
Example tokens: ['customers', 'can', 'check', 'an', 'order', 's', 'status', 'from', 'the', 'orders', 'page', 'after', 'signing', 'in', 'open', 'the', 'order', 'to', 'see', 'its', 'current', 'status', 'estimated', 'delivery', 'date']


## Cell 10 — Define semantic, keyword, and hybrid retrieval



In [10]:
def semantic_search(query, top_k=5):
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(max(1, int(top_k)), len(texts))
    scores, indices = faiss_index.search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        item = chunks_df.iloc[int(idx)].to_dict()
        item["retrieval_method"] = "semantic"
        item["semantic_score"] = float(score)
        item["rank"] = len(results) + 1
        results.append(item)
    return results


def keyword_search(query, top_k=5):
    query_tokens = tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    indices = np.argsort(scores)[::-1][:min(max(1, int(top_k)), len(texts))]
    results = []
    for idx in indices:
        score = float(scores[int(idx)])
        if score <= 0:
            continue
        item = chunks_df.iloc[int(idx)].to_dict()
        item["retrieval_method"] = "keyword"
        item["bm25_score"] = score
        item["rank"] = len(results) + 1
        results.append(item)
    return results


def hybrid_search(query, top_k=5, candidate_k=10, rrf_k=60):
    semantic_results = semantic_search(query, top_k=min(candidate_k, len(texts)))
    keyword_results = keyword_search(query, top_k=min(candidate_k, len(texts)))

    fused_scores = {}
    item_by_id = {}

    for result_list in (semantic_results, keyword_results):
        for rank, item in enumerate(result_list, start=1):
            item_id = str(item[ID_FIELD])
            fused_scores[item_id] = fused_scores.get(item_id, 0.0) + 1.0 / (rrf_k + rank)
            item_by_id[item_id] = item

    ranked_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)
    final_results = []
    for rank, item_id in enumerate(ranked_ids[:min(max(1, int(top_k)), len(ranked_ids))], start=1):
        item = dict(item_by_id[item_id])
        item["rrf_score"] = fused_scores[item_id]
        item["hybrid_rank"] = rank
        final_results.append(item)
    return final_results

print("Semantic, keyword, and hybrid retrieval functions are ready.")

Semantic, keyword, and hybrid retrieval functions are ready.


## Cell 11 — Test semantic search

In [11]:
sample_query = "How long do I have to return something I bought?"
semantic_results = semantic_search(sample_query, top_k=5)

for result in semantic_results:
    print(f"Rank {result['rank']} | similarity={result['semantic_score']:.4f} | ID={result[ID_FIELD]}")
    print(result[TEXT_FIELD][:350], "\n")

Rank 1 | similarity=0.7465 | ID=POL002_CHUNK001
Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-sale or customized items may not qualify. Customers should start a return from the order details page and follow the instructions provided. The return request mus 

Rank 2 | similarity=0.5672 | ID=FAQ004_CHUNK001
After a return is received and inspected, an approved refund is sent to the original payment method. The time it takes for the funds to appear depends on the bank or payment provider and may take 5 to 10 business days. If the refund has not appeared after that period, contact support with the order number and refund confirmation. 

Rank 3 | similarity=0.4199 | ID=FAQ005_CHUNK001
If an item arrives damaged or defective, contact support within 7 days of delivery. Include the order number, a short description of the issue, and clear photos of the i

## Cell 12 — Test keyword search

In [12]:
keyword_results = keyword_search(sample_query, top_k=5)
if not keyword_results:
    print("No keyword-overlap results were found for this wording. Try a query with exact policy terms.")
for result in keyword_results:
    print(f"Rank {result['rank']} | BM25={result['bm25_score']:.4f} | ID={result[ID_FIELD]}")
    print(result[TEXT_FIELD][:350], "\n")

Rank 1 | BM25=4.6461 | ID=FAQ002_CHUNK001
An order can usually be cancelled while it is still in the Processing status. Open Orders, select the order, and choose Cancel Order if the option is available. Orders that have already shipped cannot normally be cancelled; customers may request a return after delivery according to the returns policy. 

Rank 2 | BM25=2.2443 | ID=FAQ004_CHUNK001
After a return is received and inspected, an approved refund is sent to the original payment method. The time it takes for the funds to appear depends on the bank or payment provider and may take 5 to 10 business days. If the refund has not appeared after that period, contact support with the order number and refund confirmation. 

Rank 3 | BM25=2.0214 | ID=POL002_CHUNK001
Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-sale or customized items may not qualify. Custo

## Cell 13 — Test hybrid retrieval

In [13]:
hybrid_results = hybrid_search(sample_query, top_k=5, candidate_k=10)

for result in hybrid_results:
    print(f"Hybrid rank {result['hybrid_rank']} | RRF={result['rrf_score']:.5f} | ID={result[ID_FIELD]}")
    print(result[TEXT_FIELD][:500], "\n")

Hybrid rank 1 | RRF=0.03227 | ID=POL002_CHUNK001
Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-sale or customized items may not qualify. Customers should start a return from the order details page and follow the instructions provided. The return request must be approved before an item is sent back. 

Hybrid rank 2 | RRF=0.03226 | ID=FAQ004_CHUNK001
After a return is received and inspected, an approved refund is sent to the original payment method. The time it takes for the funds to appear depends on the bank or payment provider and may take 5 to 10 business days. If the refund has not appeared after that period, contact support with the order number and refund confirmation. 

Hybrid rank 3 | RRF=0.03202 | ID=FAQ002_CHUNK001
An order can usually be cancelled while it is still in the Processing status. Open Orders, select the order, and choose Canc

## Cell 14 — Evaluate retrieval on example customer questions

In [14]:
test_queries = [
    "How do I track my order?",
    "What is the refund policy?",
    "I cannot log in to my account.",
    "Which payment methods can I use?",
    "How is my personal information protected?"
]

for query in test_queries:
    print("=" * 90)
    print("QUESTION:", query)
    results = hybrid_search(query, top_k=3, candidate_k=10)
    for item in results:
        title = item.get("title", "Untitled")
        print(f"  [{item['hybrid_rank']}] {title} | {item[ID_FIELD]} | RRF={item['rrf_score']:.5f}")
        print("     ", item[TEXT_FIELD][:260].replace("\n", " "))

QUESTION: How do I track my order?
  [1] Contact customer support | FAQ009_CHUNK001 | RRF=0.03178
      Customers can contact support through the Help or Contact Us section of the website. Include a clear description of the problem and the relevant order number when applicable. Do not include sensitive authentication or payment information. Response times can va
  [2] Track an order | FAQ001_CHUNK001 | RRF=0.03132
      Customers can check an order's status from the Orders page after signing in. Open the order to see its current status, estimated delivery date, and tracking link when available. If tracking has not updated for more than 48 hours, contact support with the order
  [3] Item arrived damaged | FAQ005_CHUNK001 | RRF=0.03105
      If an item arrives damaged or defective, contact support within 7 days of delivery. Include the order number, a short description of the issue, and clear photos of the item and packaging when possible. Do not include passwords or full card numbers in

## Cell 15 — Try your own query

In [15]:
query = "My order arrived damaged. What should I do?"
TOP_K = 5

interactive_results = hybrid_search(query, top_k=TOP_K, candidate_k=10)
print("Question:", query)
print("Number of results:", len(interactive_results))

for item in interactive_results:
    print("\n" + "-" * 80)
    print("Rank:", item["hybrid_rank"], "| Chunk ID:", item[ID_FIELD], "| RRF:", round(item["rrf_score"], 6))
    for key in ("title", "category", "source"):
        if key in item:
            print(f"{key.title()}: {item[key]}")
    print("Text:", item[TEXT_FIELD])

Question: My order arrived damaged. What should I do?
Number of results: 5

--------------------------------------------------------------------------------
Rank: 1 | Chunk ID: FAQ005_CHUNK001 | RRF: 0.032522
Title: Item arrived damaged
Category: returns
Source: sample_faq
Text: If an item arrives damaged or defective, contact support within 7 days of delivery. Include the order number, a short description of the issue, and clear photos of the item and packaging when possible. Do not include passwords or full card numbers in the message. Support will explain the next steps after reviewing the request.

--------------------------------------------------------------------------------
Rank: 2 | Chunk ID: POL002_CHUNK001 | RRF: 0.032266
Title: Returns policy
Category: returns
Source: sample_policy
Text: Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-s

## Cell 16 — Save the FAISS index, embeddings, and mapping



In [16]:
VECTOR_DIR = PROJECT_ROOT / "data" / "vector_store"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

FAISS_PATH = VECTOR_DIR / "customer_support.faiss"
EMBEDDINGS_PATH = VECTOR_DIR / "chunk_embeddings.npy"
MAPPING_PATH = VECTOR_DIR / "chunk_mapping.pkl"
MODEL_INFO_PATH = VECTOR_DIR / "embedding_model.txt"

faiss.write_index(faiss_index, str(FAISS_PATH))
np.save(EMBEDDINGS_PATH, embeddings)

mapping_records = [
    {
        "faiss_position": int(idx),
        "chunk_id": str(row[ID_FIELD]),
        "text_field": TEXT_FIELD
    }
    for idx, row in chunks_df.iterrows()
]

with MAPPING_PATH.open("wb") as f:
    pickle.dump(mapping_records, f)

MODEL_INFO_PATH.write_text(EMBEDDING_MODEL_NAME, encoding="utf-8")

for path in [FAISS_PATH, EMBEDDINGS_PATH, MAPPING_PATH, MODEL_INFO_PATH]:
    print(f"Saved: {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")

Saved: data\vector_store\customer_support.faiss (27,693 bytes)
Saved: data\vector_store\chunk_embeddings.npy (27,776 bytes)
Saved: data\vector_store\chunk_mapping.pkl (632 bytes)
Saved: data\vector_store\embedding_model.txt (38 bytes)


## Cell 17 — Reload and verify saved artifacts

In [17]:
reloaded_index = faiss.read_index(str(FAISS_PATH))
reloaded_embeddings = np.load(EMBEDDINGS_PATH)

with MAPPING_PATH.open("rb") as f:
    reloaded_mapping = pickle.load(f)

print("Reloaded FAISS vectors:", reloaded_index.ntotal)
print("Reloaded embeddings shape:", reloaded_embeddings.shape)
print("Mapping entries:", len(reloaded_mapping))

assert reloaded_index.ntotal == len(texts), "FAISS index count mismatch."
assert reloaded_embeddings.shape == embeddings.shape, "Embedding shape mismatch."
assert len(reloaded_mapping) == len(texts), "Mapping count mismatch."

print("\nPHASE 2 ARTIFACT CHECK SUCCESS: embeddings, FAISS index, and mapping saved and reloaded.")

Reloaded FAISS vectors: 18
Reloaded embeddings shape: (18, 384)
Mapping entries: 18

PHASE 2 ARTIFACT CHECK SUCCESS: embeddings, FAISS index, and mapping saved and reloaded.
